In [ ]:
pip install transformers

In [ ]:
pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import os, json, cv2, torch, random
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import Sam3Processor, Sam3Model
import albumentations as A
from tqdm import tqdm
import torch.nn.functional as F
from sklearn.metrics import jaccard_score
from huggingface_hub import login

# ==========================================
# 1. SETUP & AUTH
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = r"E:\Prithu\Sustain\sustain train"
IMG_DIR = os.path.join(BASE_DIR, "images")
JSON_PATH = os.path.join(BASE_DIR, "sustain_fixed.json")

TILE_SIZE = 1024
OVERLAP = 150
STRIDE = TILE_SIZE - OVERLAP
NUM_CLASSES = 2 # 1: n Rich, 2: Normal

# ==========================================
# 2. DATASET (Mapped to sustain_fixed.json)
# ==========================================
class BatterySAM3Dataset(Dataset):
    def __init__(self, json_path, img_dir, image_ids, augment=False):
        with open(json_path) as f: self.data = json.load(f)
        self.img_dir = img_dir
        self.processor = Sam3Processor.from_pretrained("facebook/sam3")
        self.augment = augment
        self.cat_to_idx = {1: 0, 2: 1} # Mapping IDs to indices

        self.tiles = []
        for img_info in self.data['images']:
            if img_info['id'] in image_ids:
                H, W = img_info['height'], img_info['width']
                for y in range(0, H - TILE_SIZE + 1, STRIDE):
                    for x in range(0, W - TILE_SIZE + 1, STRIDE):
                        self.tiles.append({'img': img_info, 'x': x, 'y': y})

        self.aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.CLAHE(clip_limit=4.0, p=0.5), 
            A.GaussNoise(p=0.3)
        ])

    def __len__(self): return len(self.tiles)

    def __getitem__(self, idx):
        tile_info = self.tiles[idx]
        img_info, x, y = tile_info['img'], tile_info['x'], tile_info['y']
        img = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, img_info['file_name'])), cv2.COLOR_BGR2RGB)
        
        tile_img = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
        tile_mask = np.zeros((NUM_CLASSES, TILE_SIZE, TILE_SIZE), dtype=np.float32)
        
        for ann in [a for a in self.data['annotations'] if a['image_id'] == img_info['id']]:
            cid = self.cat_to_idx.get(ann['category_id'], 1)
            for seg in ann['segmentation']:
                poly = (np.array(seg).reshape(-1, 2) - [x, y]).astype(np.int32)
                if np.any((poly >= 0) & (poly < TILE_SIZE)):
                    cv2.fillPoly(tile_mask[cid], [poly], 1.0)

        if self.augment:
            augmented = self.aug(image=tile_img, masks=[tile_mask[0], tile_mask[1]])
            tile_img, tile_mask = augmented['image'], np.stack(augmented['masks'])

        inputs = self.processor(images=tile_img, text="n rich particle. normal particle.", return_tensors="pt")
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["ground_truth_mask"] = torch.as_tensor(tile_mask) 
        return inputs

# ==========================================
# 3. TRAINING + VALIDATION LOOP (Updated)
# ==========================================
all_ids = list(range(1, 18))
random.shuffle(all_ids)
train_ids, val_ids = all_ids[:14], all_ids[14:]

train_loader = DataLoader(BatterySAM3Dataset(JSON_PATH, IMG_DIR, train_ids, augment=True), batch_size=1, shuffle=True)
val_loader = DataLoader(BatterySAM3Dataset(JSON_PATH, IMG_DIR, val_ids, augment=False), batch_size=1, shuffle=False)

# Use GPU 1
DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"

model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)

# Freeze all parameters first
for param in model.parameters(): 
    param.requires_grad = False

# Train mask decoder only
for param in model.mask_decoder.parameters(): 
    param.requires_grad = True

# Freeze vision encoder
for param in model.vision_encoder.parameters(): 
    param.requires_grad = False

# Make sure all remaining parameters that need training are set
for name, param in model.named_parameters():
    if param.requires_grad is not True:
        param.requires_grad = True

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

best_val_iou = 0.0

for epoch in range(25):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0
    for i, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        
        # ====== CHANGE 1: do NOT cast to float16 ======
        outputs = model(
            pixel_values=batch["pixel_values"], 
            input_ids=batch["input_ids"], 
            attention_mask=batch["attention_mask"]
        )
        
        # ====== CHANGE 2: compute loss in float32 ======
        pred_up = F.interpolate(outputs.pred_masks, size=(1024, 1024), mode='bilinear').squeeze(0).float()
        gt_mask = batch["ground_truth_mask"].squeeze(0).float().clamp(0.0, 1.0)
        
        loss = F.binary_cross_entropy_with_logits(pred_up[:NUM_CLASSES], gt_mask)
        loss.backward()
        
        if (i+1) % 4 == 0:
            optimizer.step()
            optimizer.zero_grad()
        train_loss += loss.item()

    # --- VALIDATION PHASE ---
    model.eval()
    val_ious = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(
                pixel_values=batch["pixel_values"], 
                input_ids=batch["input_ids"], 
                attention_mask=batch["attention_mask"]
            )
            
            probs = torch.sigmoid(F.interpolate(out.pred_masks, size=(1024, 1024), mode='bilinear').squeeze(0).float())
            
            p_bin = (probs[:NUM_CLASSES] > 0.5).cpu().numpy().astype(np.uint8)
            g_bin = batch["ground_truth_mask"].squeeze(0).cpu().numpy().astype(np.uint8)
            
            for c in range(NUM_CLASSES):
                val_ious.append(jaccard_score(g_bin[c].flatten(), p_bin[c].flatten(), zero_division=1.0))

    current_iou = np.mean(val_ious)
    print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | Val mIoU: {current_iou:.4f}")
    
    if current_iou > best_val_iou:
        best_val_iou = current_iou
        torch.save(model.mask_decoder.state_dict(), "best_sam3_battery.pth")
        print("⭐ Best Model Saved based on mIoU")

# After training completes:
# best_threshold = run_threshold_sweep(model, val_loader)

In [ ]:
import torch
from transformers import Sam3Processor, Sam3Model
import cv2
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os

# ================== SETTINGS ==================
DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"
TILE_SIZE = 1024
OVERLAP = 150
STRIDE = TILE_SIZE - OVERLAP
NUM_CLASSES = 2

# Separate thresholds per class
THRESHOLD_NRICH = 0.25   # N-rich particles
THRESHOLD_NORMAL = 0.40  # Normal particles

# ================== LOAD MODEL ==================
processor = Sam3Processor.from_pretrained("facebook/sam3")
model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)
model.mask_decoder.load_state_dict(torch.load("best_sam3_battery.pth", map_location=DEVICE))
model.eval()

# ================== LOAD IMAGE ==================
img_path = r"E:\Prithu\Sustain\sustain train\images\SUS23_01_Kathode_Ref1_langs_1000x_HF_01.jpg"
img = cv2.imread(img_path)
if img is None:
    raise FileNotFoundError(f"Image not found or path is wrong: {img_path}")

img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
H, W, _ = img.shape

# ================== FULL-SIZE PREDICTION ARRAYS ==================
full_probs = np.zeros((NUM_CLASSES, H, W), dtype=np.float32)
count_map = np.zeros((H, W), dtype=np.float32)

# ================== SLIDING WINDOW PREDICTION ==================
ys = list(range(0, H - TILE_SIZE + 1, STRIDE)) + [H - TILE_SIZE]
xs = list(range(0, W - TILE_SIZE + 1, STRIDE)) + [W - TILE_SIZE]

for y in ys:
    for x in xs:
        tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
        inputs = processor(images=tile, text="n rich particle. normal particle.", return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k,v in inputs.items()}

        with torch.no_grad():
            outputs = model(pixel_values=inputs["pixel_values"],
                            input_ids=inputs["input_ids"],
                            attention_mask=inputs["attention_mask"])
            pred_tile = F.interpolate(outputs.pred_masks, size=(TILE_SIZE, TILE_SIZE), mode='bilinear').squeeze(0)
            probs_tile = torch.sigmoid(pred_tile)[:NUM_CLASSES].cpu().numpy()

        full_probs[:, y:y+TILE_SIZE, x:x+TILE_SIZE] += probs_tile
        count_map[y:y+TILE_SIZE, x:x+TILE_SIZE] += 1.0

# ================== AVERAGE OVERLAPS ==================
full_probs /= np.maximum(count_map, 1.0)

# ================== BINARIZE WITH SEPARATE THRESHOLDS ==================
binary_masks = np.zeros_like(full_probs, dtype=np.uint8)
binary_masks[0] = (full_probs[0] > THRESHOLD_NRICH).astype(np.uint8)  # N-rich
binary_masks[1] = (full_probs[1] > THRESHOLD_NORMAL).astype(np.uint8) # Normal

# ================== VISUALIZE ==================
plt.figure(figsize=(15,8))
plt.subplot(1,3,1)
plt.imshow(img)
plt.title("Original Image")

plt.subplot(1,3,2)
plt.imshow(binary_masks[0], cmap="Reds")
plt.title(f"N-rich Particles (>{THRESHOLD_NRICH})")

plt.subplot(1,3,3)
plt.imshow(binary_masks[1], cmap="Blues")
plt.title(f"Normal Particles (>{THRESHOLD_NORMAL})")
plt.show()

# Overlay masks
overlay = img.copy()
overlay[binary_masks[0]==1] = [255,0,0]   # Red = N-rich
overlay[binary_masks[1]==1] = [0,0,255]   # Blue = Normal

plt.figure(figsize=(10,10))
plt.imshow(overlay)
plt.title("Overlay Predicted Masks")
plt.show()

In [ ]:
import torch
from transformers import Sam3Processor, Sam3Model
import cv2
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ================== SETTINGS ==================
DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"
TILE_SIZE = 1024
OVERLAP = 150
STRIDE = TILE_SIZE - OVERLAP
NUM_CLASSES = 2
THRESHOLD = 0.01  # can adjust after threshold sweep

# ================== LOAD MODEL ==================
processor = Sam3Processor.from_pretrained("facebook/sam3")
model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)
model.mask_decoder.load_state_dict(torch.load("best_sam3_battery.pth", map_location=DEVICE))
model.eval()

# ================== INFERENCE FUNCTION ==================
def predict_mask(image_path, display_overlay=True):
    # Load image
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    H, W, _ = img.shape
    
    # Prepare empty arrays for prediction
    full_probs = np.zeros((NUM_CLASSES, H, W), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)
    
    # Sliding window prediction
    for y in range(0, H - TILE_SIZE + 1, STRIDE):
        for x in range(0, W - TILE_SIZE + 1, STRIDE):
            tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
            inputs = processor(images=tile, text="n rich particle. normal particle.", return_tensors="pt")
            inputs = {k: v.to(DEVICE) for k,v in inputs.items()}
            
            with torch.no_grad():
                outputs = model(pixel_values=inputs["pixel_values"], 
                                input_ids=inputs["input_ids"], 
                                attention_mask=inputs["attention_mask"])
                pred_tile = F.interpolate(outputs.pred_masks, size=(TILE_SIZE, TILE_SIZE), mode='bilinear').squeeze(0)
                probs_tile = torch.sigmoid(pred_tile)[:NUM_CLASSES].cpu().numpy()
            
            full_probs[:, y:y+TILE_SIZE, x:x+TILE_SIZE] += probs_tile
            count_map[y:y+TILE_SIZE, x:x+TILE_SIZE] += 1.0
    
    # Average overlapping areas
    full_probs /= np.maximum(count_map, 1.0)
    
    # Binarize masks
    binary_masks = (full_probs > THRESHOLD).astype(np.uint8)
    
    # Display overlay if requested
    if display_overlay:
        overlay = img.copy()
        overlay[binary_masks[0]==1] = [255,0,0]   # Red for n-rich
        overlay[binary_masks[1]==1] = [0,0,255]   # Blue for normal
        
        plt.figure(figsize=(12,12))
        plt.imshow(overlay)
        plt.title("Predicted Masks Overlay")
        plt.axis('off')
        plt.show()
    
    return binary_masks, full_probs


masks, probs = predict_mask(r"E:\Prithu\Sustain\sustain train\m2f_hybrid_dataset\train\images\train_3_4.jpg" ) # replace with your path")

In [ ]:
import os
import json
import cv2
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import Sam3Processor, Sam3Model
import albumentations as A
from tqdm import tqdm
from sklearn.metrics import jaccard_score, precision_recall_curve, auc

# ========================== SETTINGS ==========================
DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
BASE_DIR = r"E:\Prithu\Sustain\sustain train"
IMG_DIR = os.path.join(BASE_DIR, "images")
JSON_PATH = os.path.join(BASE_DIR, "sustain_fixed.json")

TILE_SIZE = 1024
TILES_PER_IMAGE = 30
TEXT_PROMPT = "normal particle."
SAVE_FOLDER = os.path.join(BASE_DIR, "processed_dataset")
os.makedirs(SAVE_FOLDER, exist_ok=True)

MODEL_PATH = os.path.join(BASE_DIR, "best_sam3_model.pth")

EPOCHS = 200
PATIENCE = 25

print(f"📡 Using Device: {DEVICE}")

# ========================== LOSS ==========================
def dice_loss(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum()
    return 1 - ((2.*intersection + smooth) / (pred.sum() + target.sum() + smooth))

def calculate_ap(gt, pred_probs):
    precision, recall, _ = precision_recall_curve(gt.flatten(), pred_probs.flatten())
    return auc(recall, precision)

# ========================== TILE GENERATION ==========================
def generate_tiles():
    if len(os.listdir(SAVE_FOLDER)) > 50:
        print("⏭️ Tiles already exist. Skipping generation.")
        return

    with open(JSON_PATH) as f:
        data = json.load(f)

    total_tiles = 0
    print("🚀 Generating tiles...")

    for img_info in data['images']:
        img_path = os.path.join(IMG_DIR, img_info['file_name'])
        if not os.path.exists(img_path):
            continue

        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        H, W, _ = img.shape

        mask_full = np.zeros((H, W), dtype=np.uint8)
        anns = [a for a in data['annotations'] if a['image_id'] == img_info['id'] and a['category_id'] == 2]

        for ann in anns:
            for seg in ann['segmentation']:
                poly = np.array(seg).reshape(-1, 2).astype(np.int32)
                cv2.fillPoly(mask_full, [poly], 1)

        count = 0
        step = TILE_SIZE // 2

        for y in range(0, H - TILE_SIZE + 1, step):
            for x in range(0, W - TILE_SIZE + 1, step):
                if count >= TILES_PER_IMAGE:
                    break

                tile_img = img[y:y+TILE_SIZE, x:x+TILE_SIZE]
                tile_mask = mask_full[y:y+TILE_SIZE, x:x+TILE_SIZE]

                if np.sum(tile_mask) > 200:
                    name = f"{img_info['id']}_{count}"
                    cv2.imwrite(os.path.join(SAVE_FOLDER, f"{name}.png"),
                                cv2.cvtColor(tile_img, cv2.COLOR_RGB2BGR))
                    cv2.imwrite(os.path.join(SAVE_FOLDER, f"{name}_mask.png"),
                                tile_mask * 255)
                    count += 1
                    total_tiles += 1

    print(f"✅ Total tiles created: {total_tiles}")

# ========================== DATASET ==========================
class BatteryDataset(Dataset):
    def __init__(self, folder, augment=True):
        self.folder = folder
        self.augment = augment
        self.processor = Sam3Processor.from_pretrained("facebook/sam3")

        self.images = [f for f in os.listdir(folder) if f.endswith(".png") and "_mask" not in f]

        self.aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomBrightnessContrast(p=0.2),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.3),
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]

        img = cv2.cvtColor(cv2.imread(os.path.join(self.folder, img_name)), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(os.path.join(self.folder, img_name.replace(".png", "_mask.png")), 0) / 255.0

        if self.augment:
            aug = self.aug(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']

        inputs = self.processor(images=img, text=TEXT_PROMPT, return_tensors="pt")
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["gt_mask"] = torch.tensor(mask).unsqueeze(0)

        return inputs

# ========================== PREP ==========================
generate_tiles()

dataset = BatteryDataset(SAVE_FOLDER)

train_size = int(0.85 * len(dataset))
train_ds = torch.utils.data.Subset(dataset, range(train_size))
val_ds = torch.utils.data.Subset(dataset, range(train_size, len(dataset)))

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

# ========================== MODEL ==========================
model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)

for p in model.vision_encoder.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(model.mask_decoder.parameters(), lr=1e-5)

# ========================== TRAIN ==========================
best_iou = 0
no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        out = model(pixel_values=batch["pixel_values"],
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"])

        pred = F.interpolate(out.pred_masks, size=(TILE_SIZE, TILE_SIZE), mode='bilinear')
        gt = batch["gt_mask"].float()

        loss = F.binary_cross_entropy_with_logits(pred[:, 0:1], gt) + dice_loss(pred[:, 0:1], gt)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # ================= VALIDATION =================
    model.eval()
    ious, aps = [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            out = model(pixel_values=batch["pixel_values"],
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"])

            prob = torch.sigmoid(F.interpolate(out.pred_masks, size=(TILE_SIZE, TILE_SIZE), mode='bilinear'))

            pred_bin = (prob[:, 0] > 0.5).cpu().numpy().astype(np.uint8)
            gt_bin = batch["gt_mask"].cpu().numpy().astype(np.uint8)

            ious.append(jaccard_score(gt_bin.flatten(), pred_bin.flatten(), zero_division=1))
            aps.append(calculate_ap(gt_bin, prob[:, 0].cpu().numpy()))

    mean_iou = np.mean(ious)
    mean_ap = np.mean(aps)

    print(f"\n📈 Epoch {epoch+1}")
    print(f"Loss: {train_loss/len(train_loader):.4f}")
    print(f"mIoU: {mean_iou:.4f} | mAP: {mean_ap:.4f}")

    # ================= SAVE BEST =================
    if mean_iou > best_iou:
        best_iou = mean_iou
        torch.save(model.mask_decoder.state_dict(), MODEL_PATH)
        print(f"⭐ Best model saved!")
        no_improve = 0
    else:
        no_improve += 1

    # ================= EARLY STOP =================
    if no_improve >= PATIENCE:
        print(f"⛔ Early stopping triggered at epoch {epoch+1}")
        break

In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn.functional as F
from transformers import Sam3Processor, Sam3Model
import matplotlib.pyplot as plt

# ========================== SETTINGS ==========================
DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"

BASE_DIR = r"E:\Prithu\Sustain\sustain train"
MODEL_PATH = os.path.join(BASE_DIR, "best_sam3_model.pth")

IMAGE_PATH = r"E:\Prithu\Sustain\sustain train\images\SUS23_01_Kathode_Ref1_langs_1000x_HF_01.jpg"   # <-- YOUR IMAGE

TILE_SIZE = 1024
TEXT_PROMPT = "normal particle."

print(f"📡 Using Device: {DEVICE}")

# ========================== LOAD MODEL ==========================
processor = Sam3Processor.from_pretrained("facebook/sam3")
model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)

# 🔥 Load your trained weights
model.mask_decoder.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print("✅ Model loaded successfully")

# ========================== LOAD IMAGE ==========================
img = cv2.cvtColor(cv2.imread(IMAGE_PATH), cv2.COLOR_BGR2RGB)
H, W, _ = img.shape

# ========================== TILE INFERENCE ==========================
pred_mask_full = np.zeros((H, W), dtype=np.float32)

step = TILE_SIZE // 2

for y in range(0, H - TILE_SIZE + 1, step):
    for x in range(0, W - TILE_SIZE + 1, step):

        tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]

        inputs = processor(images=tile, text=TEXT_PROMPT, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            out = model(pixel_values=inputs["pixel_values"],
                        input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"])

        prob = torch.sigmoid(
            F.interpolate(out.pred_masks, size=(TILE_SIZE, TILE_SIZE), mode='bilinear')
        )[0, 0].cpu().numpy()

        # Merge tiles (max blending)
        pred_mask_full[y:y+TILE_SIZE, x:x+TILE_SIZE] = np.maximum(
            pred_mask_full[y:y+TILE_SIZE, x:x+TILE_SIZE],
            prob
        )

# ========================== THRESHOLD ==========================
THRESHOLD = 0.5   # you can tune this
binary_mask = (pred_mask_full > THRESHOLD).astype(np.uint8)

# ========================== VISUALIZATION ==========================
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.title("Original")
plt.imshow(img)

plt.subplot(1,3,2)
plt.title("Prediction Heatmap")
plt.imshow(pred_mask_full, cmap='jet')

plt.subplot(1,3,3)
plt.title("Binary Mask")
plt.imshow(binary_mask, cmap='gray')

plt.show()

# ========================== SAVE OUTPUT ==========================
cv2.imwrite(os.path.join(BASE_DIR, "prediction_mask.png"), binary_mask*255)
print("💾 Prediction saved!")

In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn.functional as F
from transformers import Sam3Processor, Sam3Model
import matplotlib.pyplot as plt

# ========================== SETTINGS ==========================
DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"

BASE_DIR = r"E:\Prithu\Sustain\sustain train"
MODEL_PATH = os.path.join(BASE_DIR, "best_sam3_model.pth")

IMAGE_PATH = r"E:\Prithu\Sustain\sustain train\images\sus24_09_kathode_acc102_1000x_hf_12.jpg"

TILE_SIZE = 1024
STEP = TILE_SIZE // 2

# 🔥 IMPORTANT (same as training)
TEXT_PROMPT = "normal particle."

THRESHOLD = 0.25

print(f"📡 Using Device: {DEVICE}")
print(f"🧠 Prompt: {TEXT_PROMPT}")

# ========================== LOAD MODEL ==========================
processor = Sam3Processor.from_pretrained("facebook/sam3")
model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)

model.mask_decoder.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print("✅ Model loaded")

# ========================== LOAD IMAGE ==========================
img = cv2.cvtColor(cv2.imread(IMAGE_PATH), cv2.COLOR_BGR2RGB)
H, W, _ = img.shape

# ========================== HELP FUNCTION ==========================
def get_positions(size, tile_size, step):
    positions = list(range(0, size - tile_size + 1, step))
    if positions[-1] != size - tile_size:
        positions.append(size - tile_size)
    return positions

ys = get_positions(H, TILE_SIZE, STEP)
xs = get_positions(W, TILE_SIZE, STEP)

# ========================== TILE PREDICTION ==========================
pred_mask_full = np.zeros((H, W), dtype=np.float32)
count_map = np.zeros((H, W), dtype=np.float32)  # for smooth blending

for y in ys:
    for x in xs:

        tile = img[y:y+TILE_SIZE, x:x+TILE_SIZE]

        # 🔥 Padding for edge tiles
        if tile.shape[0] != TILE_SIZE or tile.shape[1] != TILE_SIZE:
            pad_h = TILE_SIZE - tile.shape[0]
            pad_w = TILE_SIZE - tile.shape[1]

            tile = cv2.copyMakeBorder(
                tile, 0, pad_h, 0, pad_w,
                cv2.BORDER_CONSTANT, value=0
            )

        inputs = processor(images=tile, text=TEXT_PROMPT, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            out = model(pixel_values=inputs["pixel_values"],
                        input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"])

        prob = torch.sigmoid(
            F.interpolate(out.pred_masks, size=(TILE_SIZE, TILE_SIZE), mode='bilinear')
        )[0, 0].cpu().numpy()

        # Remove padding effect
        h_valid = min(TILE_SIZE, H - y)
        w_valid = min(TILE_SIZE, W - x)

        pred_mask_full[y:y+h_valid, x:x+w_valid] += prob[:h_valid, :w_valid]
        count_map[y:y+h_valid, x:x+w_valid] += 1

# 🔥 Smooth averaging (better than max)
pred_mask_full /= count_map

# ========================== POST PROCESS ==========================
binary_mask = (pred_mask_full > THRESHOLD).astype(np.uint8)

# ========================== OVERLAY ==========================
colored_mask = np.zeros_like(img)
colored_mask[:, :, 0] = binary_mask * 255  # RED

overlay = cv2.addWeighted(img, 1.0, colored_mask, 0.5, 0)

# ========================== VISUALIZATION ==========================
plt.figure(figsize=(20,5))

plt.subplot(1,4,1)
plt.title("Original")
plt.imshow(img)
plt.axis("off")

plt.subplot(1,4,2)
plt.title("Heatmap")
plt.imshow(pred_mask_full, cmap='jet')
plt.axis("off")

plt.subplot(1,4,3)
plt.title("Mask")
plt.imshow(binary_mask, cmap='gray')
plt.axis("off")

plt.subplot(1,4,4)
plt.title("Overlay")
plt.imshow(overlay)
plt.axis("off")

plt.tight_layout()
plt.show()

# ========================== SAVE ==========================
cv2.imwrite(os.path.join(BASE_DIR, "pred_mask.png"), binary_mask*255)
cv2.imwrite(os.path.join(BASE_DIR, "overlay.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

print("💾 Saved outputs successfully")

In [ ]:
import os
import json
import cv2
import torch
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import Sam3Processor, Sam3Model
import albumentations as A
from tqdm import tqdm
import gc
import random
from scipy.optimize import linear_sum_assignment

# ========================== SETTINGS ==========================
DEVICE = torch.device("cuda:0" if torch.cuda.device_count() >= 2 else "cuda:1")
BASE_DIR = r"E:\Prithu\Sustain Image with COCO JSON"
IMG_DIR = os.path.join(BASE_DIR, "images")
JSON_PATH = os.path.join(BASE_DIR, "sustain_fixed.json")
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "sam3_instance_fullres.pth")

EPOCHS = 200
LEARNING_RATE = 1e-5
MAX_BOXES_PER_BATCH = 64  

# ========================== DATASET ==========================
class SamFullResDataset(Dataset):
    def __init__(self, json_path, img_dir, processor):
        self.img_dir = img_dir
        self.processor = processor
        
        with open(json_path) as f:
            data = json.load(f)
        
        self.img_to_anns = {}
        for ann in data['annotations']:
            # --- THE FIX: Removed the strict category_id check ---
            # This ensures all annotated images are loaded, regardless of label number.
            self.img_to_anns.setdefault(ann['image_id'], []).append(ann)

        # Filter to ONLY images that have labeled particles
        self.images = [img for img in data['images'] if img['id'] in self.img_to_anns]
        print(f"Loaded {len(self.images)} valid images from the JSON/Folder.")

        # --- VALID, SAFE AUGMENTATION ---
        self.aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomBrightnessContrast(p=0.2),
        ], bbox_params=A.BboxParams(format='coco', label_fields=['category_ids'], min_visibility=0.2, min_area=10))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.img_dir, img_info['file_name'])
        
        image = cv2.imread(img_path)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {img_path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]

        anns = self.img_to_anns.get(img_info['id'], [])
        
        bboxes, masks, category_ids = [], [], []

        for ann in anns:
            # --- PRE-CLIPPING TO PREVENT ALBUMENTATIONS CRASH ---
            x, y, bw, bh = ann['bbox']
            
            # Snap coordinates strictly to the edges of the image
            x_min = max(0, float(x))
            y_min = max(0, float(y))
            x_max = min(float(w), float(x + bw))
            y_max = min(float(h), float(y + bh))
            
            new_w = x_max - x_min
            new_h = y_max - y_min
            
            # If the box is completely invalid or smaller than 1 pixel, skip it safely
            if new_w <= 1 or new_h <= 1:
                continue

            bboxes.append([x_min, y_min, new_w, new_h]) 
            category_ids.append(ann['category_id'])
            
            mask = np.zeros((h, w), dtype=np.uint8)
            for seg in ann['segmentation']:
                poly = np.array(seg).reshape(-1, 2).astype(np.int32)
                cv2.fillPoly(mask, [poly], 1)
            masks.append(mask)

        # Apply Safe Augmentation
        transformed = self.aug(image=image, bboxes=bboxes, masks=masks, category_ids=category_ids)
        image = transformed['image']
        aug_bboxes = transformed['bboxes']
        aug_masks = transformed['masks']

        # --- INSTANCE SUB-SAMPLING ---
        # Prevents VRAM crashes if an image has hundreds of particles
        if len(aug_bboxes) > MAX_BOXES_PER_BATCH:
            indices = random.sample(range(len(aug_bboxes)), MAX_BOXES_PER_BATCH)
            aug_bboxes = [aug_bboxes[i] for i in indices]
            aug_masks = [aug_masks[i] for i in indices]

        # Convert COCO [x,y,w,h] to SAM format [x1,y1,x2,y2]
        sam_boxes = []
        for b in aug_bboxes:
            x1, y1 = max(0, b[0]), max(0, b[1])
            x2, y2 = min(w, b[0] + b[2]), min(h, b[1] + b[3])
            sam_boxes.append([x1, y1, x2, y2])

        if len(sam_boxes) == 0:
            return self.__getitem__((idx + 1) % self.__len__())

        # Forward through processor with mandatory text token
        inputs = self.processor(
            image, 
            text="particle.", 
            input_boxes=[sam_boxes], 
            return_tensors="pt"
        )
        
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["gt_masks"] = torch.tensor(np.array(aug_masks)).float()

        return inputs

# ========================== MATCHING, LOSS, AND METRICS ==========================
def calc_loss_and_metrics(pred_masks, gt_masks):
    # 1. Dynamically read SAM's output shape
    pred_h, pred_w = pred_masks.shape[-2:]
    
    # 2. Downscale GT to match model output perfectly
    gt_masks_scaled = F.interpolate(gt_masks, size=(pred_h, pred_w), mode="bilinear", align_corners=False)
    
    pred = pred_masks[0] # [200, pred_h, pred_w]
    gt = gt_masks_scaled[0] # [N, pred_h, pred_w]
    
    # 3. Flatten tensors
    pred_flat = pred.flatten(1) # [200, H*W]
    gt_flat = gt.flatten(1)     # [N, H*W]
    
    # 4. Hungarian Matching for DETR-style heads
    with torch.no_grad():
        pred_prob = torch.sigmoid(pred_flat)
        inter = torch.mm(pred_prob, gt_flat.T) 
        union = pred_prob.sum(1, keepdim=True) + gt_flat.sum(1, keepdim=True).T
        cost_matrix = - (2 * inter + 1e-6) / (union + 1e-6) 
        
        pred_idx, gt_idx = linear_sum_assignment(cost_matrix.cpu().numpy())
        
    # 5. Extract Matched Pairs
    matched_preds = pred_flat[pred_idx]
    matched_gts = gt_flat[gt_idx]
    
    # 6. Calculate Losses
    bce = F.binary_cross_entropy_with_logits(matched_preds, matched_gts)
    
    matched_probs = torch.sigmoid(matched_preds)
    inter_loss = (matched_probs * matched_gts).sum(dim=1)
    union_loss = matched_probs.sum(dim=1) + matched_gts.sum(dim=1)
    dice = 1 - ((2. * inter_loss + 1e-6) / (union_loss + 1e-6))
    
    total_loss = bce + dice.mean()

    # 7. Calculate Metrics (IoU, Precision, Recall)
    with torch.no_grad():
        pred_bin = (matched_probs > 0.5).float()
        tp = (pred_bin * matched_gts).sum(dim=1)
        fp = (pred_bin * (1 - matched_gts)).sum(dim=1)
        fn = ((1 - pred_bin) * matched_gts).sum(dim=1)

        iou = tp / (tp + fp + fn + 1e-6)
        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)

    metrics = {
        "loss": total_loss,
        "iou": iou.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item()
    }
    
    return metrics

# ========================== MAIN TRAINING ==========================
def train():
    gc.collect()
    torch.cuda.empty_cache()

    print(f"Loading SAM 3 Processor and Model on {DEVICE}...")
    processor = Sam3Processor.from_pretrained("facebook/sam3")
    model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)

    # Freeze Vision Encoder to save massive amounts of VRAM
    for p in model.vision_encoder.parameters():
        p.requires_grad = False

    dataset = SamFullResDataset(JSON_PATH, IMG_DIR, processor)
    train_loader = DataLoader(dataset, batch_size=1, shuffle=True)

    optimizer = torch.optim.AdamW(model.mask_decoder.parameters(), lr=LEARNING_RATE)
    
    # --- DYNAMIC LEARNING RATE SCHEDULER ---
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=10
    )

    print(f"🚀 Training Started!")

    best_iou = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        
        epoch_loss = 0
        epoch_iou = 0
        epoch_prec = 0
        epoch_rec = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for batch in pbar:
            pixel_values = batch["pixel_values"].to(DEVICE)
            input_boxes = batch["input_boxes"].to(DEVICE)
            input_ids = batch["input_ids"].to(DEVICE)          
            attention_mask = batch["attention_mask"].to(DEVICE) 
            gt_masks = batch["gt_masks"].to(DEVICE) 
            
            outputs = model(
                pixel_values=pixel_values,
                input_boxes=input_boxes,
                input_ids=input_ids,             
                attention_mask=attention_mask,   
                multimask_output=False
            )

            pred_masks = outputs.pred_masks.squeeze(2) 
            
            # Pass directly to dynamic matching logic
            metrics = calc_loss_and_metrics(pred_masks, gt_masks)
            loss = metrics["loss"]

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Track metrics
            epoch_loss += loss.item()
            epoch_iou += metrics["iou"]
            epoch_prec += metrics["precision"]
            epoch_rec += metrics["recall"]
            
            pbar.set_postfix({
                "Loss": f"{loss.item():.3f}", 
                "IoU": f"{metrics['iou']:.3f}",
                "Prec": f"{metrics['precision']:.3f}"
            })

        # Calculate epoch averages
        avg_loss = epoch_loss / len(train_loader)
        avg_iou = epoch_iou / len(train_loader)
        avg_prec = epoch_prec / len(train_loader)
        avg_rec = epoch_rec / len(train_loader)
        
        print(f"\n📊 Epoch {epoch+1} Summary:")
        print(f"   Loss: {avg_loss:.4f} | IoU: {avg_iou:.4f} | Precision: {avg_prec:.4f} | Recall: {avg_rec:.4f}")
        print(f"   Current LR: {optimizer.param_groups[0]['lr']:.8f}")
        
        # Step the LR Scheduler based on IoU performance
        scheduler.step(avg_iou)
        
        # Save based on best IoU 
        if avg_iou > best_iou:
            best_iou = avg_iou
            torch.save(model.mask_decoder.state_dict(), MODEL_SAVE_PATH)
            print(f"⭐ New Best IoU: {best_iou:.4f} - Model Saved successfully!\n")
        else:
            print() # Blank line for formatting readability

if __name__ == "__main__":
    train()

In [ ]:
import os
import json
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
import albumentations as A
from albumentations.pytorch import ToTensorV2

cv2.setNumThreads(0)

# ==============================
# PATHS
# ==============================
base_dir = r"E:\Prithu\Sustain Image with COCO JSON"
img_dir = os.path.join(base_dir, "images")
ann_file = os.path.join(base_dir, "sustain_fixed.json")

device = torch.device("cuda:1" if torch.cuda.device_count() >= 2 else "cuda:0")

# ==============================
# AUGMENTATION
# ==============================
train_tfms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.85,1.15), rotate=(-25,25), translate_percent=(0.05,0.05), p=0.6),
    A.OneOf([
        A.GaussNoise(p=1),
        A.RandomBrightnessContrast(p=1),
        A.HueSaturationValue(p=1)
    ], p=0.5),
    A.Normalize(),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

val_tfms = A.Compose([
    A.Normalize(),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

# ==============================
# DATASET
# ==============================
class CocoDataset(Dataset):
    def __init__(self, img_dir, ann_file, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms

        with open(ann_file) as f:
            coco = json.load(f)

        self.images = {img['id']: img for img in coco['images']}
        self.cat_map = {cat['id']: i+1 for i, cat in enumerate(coco['categories'])}

        self.img_to_anns = {}
        for ann in coco['annotations']:
            self.img_to_anns.setdefault(ann['image_id'], []).append(ann)

        self.ids = list(self.images.keys())

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        info = self.images[img_id]

        path = os.path.join(self.img_dir, info['file_name'])
        img = cv2.imread(path)
        if img is None:
            raise FileNotFoundError(path)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        anns = self.img_to_anns.get(img_id, [])

        boxes, labels, masks = [], [], []

        for ann in anns:
            x, y, bw, bh = ann['bbox']
            x1, y1 = max(0,x), max(0,y)
            x2, y2 = min(w,x+bw), min(h,y+bh)

            if (x2-x1) <= 1 or (y2-y1) <= 1:
                continue

            boxes.append([x1,y1,x2,y2])
            labels.append(self.cat_map[ann['category_id']])

            mask = np.zeros((h,w), dtype=np.uint8)
            for seg in ann['segmentation']:
                pts = np.array(seg).reshape(-1,2).astype(np.int32)
                cv2.fillPoly(mask, [pts], 1)
            masks.append(mask)

        if self.transforms:
            t = self.transforms(image=img, bboxes=boxes, masks=masks, labels=labels)
            img = t['image']
            boxes = t['bboxes']
            masks = t['masks']
            labels = t['labels']

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0,4)),
            "labels": torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,),dtype=torch.int64),
            "masks": torch.tensor(np.stack(masks), dtype=torch.uint8) if len(masks)>0 else torch.zeros((0,h,w),dtype=torch.uint8)
        }

        return img, target

    def __len__(self):
        return len(self.ids)

def collate_fn(batch):
    return tuple(zip(*batch))

# ==============================
# DATA SPLIT
# ==============================
full_ds = CocoDataset(img_dir, ann_file)
n = len(full_ds)
train_size = int(0.8*n)

train_ds = torch.utils.data.Subset(
    CocoDataset(img_dir, ann_file, transforms=train_tfms),
    list(range(train_size))
)

val_ds = torch.utils.data.Subset(
    CocoDataset(img_dir, ann_file, transforms=val_tfms),
    list(range(train_size, n))
)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)

# ==============================
# MODEL
# ==============================
num_classes = len(full_ds.cat_map)+1

model = maskrcnn_resnet50_fpn(weights="DEFAULT")

model.rpn.anchor_generator = AnchorGenerator(
    ((8,), (16,), (32,), (64,), (128,)),
    ((0.5,1.0,2.0),)*5
)

in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

in_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
model.roi_heads.mask_predictor = MaskRCNNPredictor(in_mask,256,num_classes)

model.to(device)

# ==============================
# TRAINING
# ==============================
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

best_iou = 0
patience = 25
counter = 0
history = []

for epoch in range(200):
    model.train()
    total_loss = 0

    for imgs, targets in train_loader:
        imgs = [i.to(device) for i in imgs]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]

        loss = sum(model(imgs, targets).values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # ======================
    # VALIDATION METRICS
    # ======================
    model.eval()

    ious, precisions, recalls, accs = [], [], [], []

    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = [i.to(device) for i in imgs]
            preds = model(imgs)

            for i in range(len(imgs)):
                if len(preds[i]['masks']) == 0:
                    continue

                pred_mask = (preds[i]['masks'].squeeze(1).sum(0) > 0.5).cpu()
                true_mask = (targets[i]['masks'].sum(0) > 0).cpu()

                tp = ((pred_mask==1)&(true_mask==1)).sum().item()
                tn = ((pred_mask==0)&(true_mask==0)).sum().item()
                fp = ((pred_mask==1)&(true_mask==0)).sum().item()
                fn = ((pred_mask==0)&(true_mask==1)).sum().item()

                iou = tp/(tp+fp+fn+1e-6)
                precision = tp/(tp+fp+1e-6)
                recall = tp/(tp+fn+1e-6)
                acc = (tp+tn)/(tp+tn+fp+fn+1e-6)

                ious.append(iou)
                precisions.append(precision)
                recalls.append(recall)
                accs.append(acc)

    miou = np.mean(ious) if ious else 0
    mprec = np.mean(precisions) if precisions else 0
    mrec = np.mean(recalls) if recalls else 0
    macc = np.mean(accs) if accs else 0

    print(f"\nEpoch {epoch+1}")
    print(f"Loss: {total_loss:.4f}")
    print(f"IoU: {miou:.4f} | Precision: {mprec:.4f} | Recall: {mrec:.4f} | Accuracy: {macc:.4f}")

    history.append(miou)

    if miou > best_iou:
        best_iou = miou
        counter = 0
        torch.save(model.state_dict(), os.path.join(base_dir,"best_model.pth"))
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

# ==============================
# PLOT
# ==============================
plt.plot(history)
plt.title("IoU vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.grid()
plt.show()

print("✅ Training complete. Best model saved.")

In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn.functional as F
from transformers import Sam3Processor, Sam3Model
import matplotlib.pyplot as plt

# ========================== SETTINGS ==========================
DEVICE = "cuda:0"  # Forced to use GPU 0

BASE_DIR = r"E:\Prithu\Sustain\sustain train"
MODEL_PATH = os.path.join(BASE_DIR, "best_sam3_model.pth")

IMAGE_PATH = r"E:\Prithu\Sustain\sustain train\images\sus24_09_kathode_acc102_1000x_hf_12.jpg"

TILE_SIZE = 1024
STEP = TILE_SIZE // 2

# 🔥 IMPORTANT (same as training)
TEXT_PROMPT = "normal particle."

THRESHOLD = 0.25

print(f"📡 Using Device: {DEVICE}")
print(f"🧠 Prompt: {TEXT_PROMPT}")

# ========================== LOAD MODEL ==========================
processor = Sam3Processor.from_pretrained("facebook/sam3")
model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)

model.mask_decoder.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print("✅ Model loaded")

# ========================== LOAD IMAGE ==========================
img = cv2.imread(IMAGE_PATH)
if img is None:
    raise ValueError(f"Could not load image at {IMAGE_PATH}")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
H, W, _ = img_rgb.shape

# ========================== HELP FUNCTION ==========================
def get_positions(size, tile_size, step):
    positions = list(range(0, size - tile_size + 1, step))
    if not positions or positions[-1] != size - tile_size:
        positions.append(max(0, size - tile_size))
    return positions

ys = get_positions(H, TILE_SIZE, STEP)
xs = get_positions(W, TILE_SIZE, STEP)

# ========================== TILE PREDICTION ==========================
pred_mask_full = np.zeros((H, W), dtype=np.float32)
count_map = np.zeros((H, W), dtype=np.float32)  # for smooth blending

for y in ys:
    for x in xs:
        tile = img_rgb[y:y+TILE_SIZE, x:x+TILE_SIZE]

        # 🔥 Padding for edge tiles
        if tile.shape[0] != TILE_SIZE or tile.shape[1] != TILE_SIZE:
            pad_h = TILE_SIZE - tile.shape[0]
            pad_w = TILE_SIZE - tile.shape[1]

            tile = cv2.copyMakeBorder(
                tile, 0, pad_h, 0, pad_w,
                cv2.BORDER_CONSTANT, value=0
            )

        inputs = processor(images=tile, text=TEXT_PROMPT, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            out = model(pixel_values=inputs["pixel_values"],
                        input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"])

        prob = torch.sigmoid(
            F.interpolate(out.pred_masks, size=(TILE_SIZE, TILE_SIZE), mode='bilinear')
        )[0, 0].cpu().numpy()

        # Remove padding effect
        h_valid = min(TILE_SIZE, H - y)
        w_valid = min(TILE_SIZE, W - x)

        pred_mask_full[y:y+h_valid, x:x+w_valid] += prob[:h_valid, :w_valid]
        count_map[y:y+h_valid, x:x+w_valid] += 1

# 🔥 Smooth averaging (better than max)
pred_mask_full /= count_map

# ========================== POST PROCESS ==========================
binary_mask = (pred_mask_full > THRESHOLD).astype(np.uint8)

# ========================== OVERLAY ==========================
colored_mask = np.zeros_like(img_rgb)
colored_mask[:, :, 0] = binary_mask * 255  # RED

overlay = cv2.addWeighted(img_rgb, 1.0, colored_mask, 0.5, 0)

# ========================== SAVE & VISUALIZATION ==========================
# Save original high-resolution image to disk
overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
cv2.imwrite(os.path.join(BASE_DIR, "overlay_prediction.png"), overlay_bgr)
print("💾 Saved high-quality output successfully")

# Display ONLY the final overlaid image in original proportions (No titles, no axes)
dpi = 100
fig = plt.figure(figsize=(W / dpi, H / dpi), dpi=dpi)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(overlay)
plt.show()

In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn.functional as F
from transformers import Sam3Processor, Sam3Model
import matplotlib.pyplot as plt

# ========================== SETTINGS ==========================
DEVICE = "cuda:0"  # Forced to use GPU 0

BASE_DIR = r"E:\Prithu\Sustain\sustain train"
MODEL_PATH = os.path.join(BASE_DIR, "best_sam3_model.pth")

IMAGE_PATH = r"E:\Prithu\Sustain\sustain train\images\sus24_09_kathode_acc102_1000x_hf_12.jpg"

TILE_SIZE = 1024
STEP = TILE_SIZE // 2

# 🔥 IMPORTANT (same as training)
TEXT_PROMPT = "normal particle."

THRESHOLD = 0.25

print(f"📡 Using Device: {DEVICE}")
print(f"🧠 Prompt: {TEXT_PROMPT}")

# ========================== LOAD MODEL ==========================
processor = Sam3Processor.from_pretrained("facebook/sam3")
model = Sam3Model.from_pretrained("facebook/sam3").to(DEVICE)

model.mask_decoder.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print("✅ Model loaded")

# ========================== LOAD IMAGE ==========================
img = cv2.imread(IMAGE_PATH)
if img is None:
    raise ValueError(f"Could not load image at {IMAGE_PATH}")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
H, W, _ = img_rgb.shape

# ========================== HELP FUNCTION ==========================
def get_positions(size, tile_size, step):
    positions = list(range(0, size - tile_size + 1, step))
    if not positions or positions[-1] != size - tile_size:
        positions.append(max(0, size - tile_size))
    return positions

ys = get_positions(H, TILE_SIZE, STEP)
xs = get_positions(W, TILE_SIZE, STEP)

# ========================== TILE PREDICTION ==========================
pred_mask_full = np.zeros((H, W), dtype=np.float32)
count_map = np.zeros((H, W), dtype=np.float32)  # for smooth blending

for y in ys:
    for x in xs:
        tile = img_rgb[y:y+TILE_SIZE, x:x+TILE_SIZE]

        # 🔥 Padding for edge tiles
        if tile.shape[0] != TILE_SIZE or tile.shape[1] != TILE_SIZE:
            pad_h = TILE_SIZE - tile.shape[0]
            pad_w = TILE_SIZE - tile.shape[1]

            tile = cv2.copyMakeBorder(
                tile, 0, pad_h, 0, pad_w,
                cv2.BORDER_CONSTANT, value=0
            )

        inputs = processor(images=tile, text=TEXT_PROMPT, return_tensors="pt")
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        with torch.no_grad():
            out = model(pixel_values=inputs["pixel_values"],
                        input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"])

        prob = torch.sigmoid(
            F.interpolate(out.pred_masks, size=(TILE_SIZE, TILE_SIZE), mode='bilinear')
        )[0, 0].cpu().numpy()

        # Remove padding effect
        h_valid = min(TILE_SIZE, H - y)
        w_valid = min(TILE_SIZE, W - x)

        pred_mask_full[y:y+h_valid, x:x+w_valid] += prob[:h_valid, :w_valid]
        count_map[y:y+h_valid, x:x+w_valid] += 1

# 🔥 Smooth averaging (better than max)
pred_mask_full /= count_map

# ========================== POST PROCESS ==========================
binary_mask = (pred_mask_full > THRESHOLD).astype(np.uint8)

# ========================== OVERLAY ==========================
colored_mask = np.zeros_like(img_rgb)
colored_mask[:, :, 0] = binary_mask * 255  # RED

overlay = cv2.addWeighted(img_rgb, 1.0, colored_mask, 0.5, 0)

# ========================== SAVE & VISUALIZATION ==========================
# Save original high-resolution image to disk
overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
cv2.imwrite(os.path.join(BASE_DIR, "overlay_prediction.png"), overlay_bgr)
print("💾 Saved high-quality output successfully")

# Display ONLY the final overlaid image in original proportions (No titles, no axes)
dpi = 100
fig = plt.figure(figsize=(W / dpi, H / dpi), dpi=dpi)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(overlay)
plt.show()

In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# ========================== SETTINGS ==========================
DEVICE = "cuda:0"  # Forced to use GPU 0
BASE_DIR = r"E:\Prithu\Sustain\sustain train"
MODEL_PATH = os.path.join(BASE_DIR, "best_hybrid_watershed_unet.pth")

IMAGE_PATH = r"E:\Prithu\Sustain Image with COCO JSON\images\sus23_01_kathode_ref1_quer_1000x_hf_04.jpg"
PATCH_SIZE = 512
OVERLAP = 128
STRIDE = PATCH_SIZE - OVERLAP
THRESHOLD = 0.5

print(f"📡 Using Device: {DEVICE}")

# ========================== MODEL ==========================
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c)
        )
        self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1), nn.BatchNorm2d(out_c))
        
    def forward(self, x):
        return F.relu(self.conv(x) + self.shortcut(x))

class HybridWatershedUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ResidualBlock(3, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ResidualBlock(128, 256)
        self.up1 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec1 = ResidualBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = ResidualBlock(128, 64)
        self.mask_out = nn.Conv2d(64, 2, 1)   # 2 classes: n-rich + normal
        self.dist_out = nn.Conv2d(64, 1, 1)
        
    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        b = self.bottleneck(self.pool(s2))
        d1 = self.dec1(torch.cat([self.up1(b), s2], 1))
        d2 = self.dec2(torch.cat([self.up2(d1), s1], 1))
        return self.mask_out(d2), torch.sigmoid(self.dist_out(d2))

# Load trained model
model = HybridWatershedUNet().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("✅ Model Loaded")

# ========================== LOAD IMAGE ==========================
img = cv2.imread(IMAGE_PATH)
if img is None:
    raise ValueError(f"Could not load image at {IMAGE_PATH}")
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
H, W, _ = img_rgb.shape

# Prepare full-size heatmaps
nrich_map = np.zeros((H, W), dtype=np.float32)
normal_map = np.zeros((H, W), dtype=np.float32)

# ========================== SLIDING WINDOW PREDICTION ==========================
y_positions = list(range(0, H - PATCH_SIZE + 1, STRIDE))
x_positions = list(range(0, W - PATCH_SIZE + 1, STRIDE))

# Add last patch if needed to cover right/bottom edges
if not y_positions or y_positions[-1] != H - PATCH_SIZE:
    y_positions.append(max(0, H - PATCH_SIZE))
if not x_positions or x_positions[-1] != W - PATCH_SIZE:
    x_positions.append(max(0, W - PATCH_SIZE))

for y in tqdm(y_positions, desc="Sliding Window"):
    for x in x_positions:
        tile = img_rgb[y:y+PATCH_SIZE, x:x+PATCH_SIZE].astype(np.float32) / 255.0
        tile_tensor = torch.from_numpy(tile.transpose(2,0,1)).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            pred_mask, _ = model(tile_tensor)
            pred_mask = torch.sigmoid(pred_mask).cpu().numpy()[0]  # (2,H,W)

        # Merge predictions using max (to handle overlaps)
        nrich_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = np.maximum(
            nrich_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE], pred_mask[0]
        )
        normal_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = np.maximum(
            normal_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE], pred_mask[1]
        )

# ========================== THRESHOLD ==========================
nrich_bin = (nrich_map > THRESHOLD).astype(np.uint8)
normal_bin = (normal_map > THRESHOLD).astype(np.uint8)

# ========================== OVERLAY ==========================
colored = np.zeros_like(img_rgb)
colored[:, :, 0] = nrich_bin * 255   # Red channel for n-rich
colored[:, :, 1] = normal_bin * 255  # Green channel for normal

alpha = 0.5
overlay = cv2.addWeighted(img_rgb, 1.0, colored, alpha, 0)

# ========================== SAVE & VISUALIZATION ==========================
# Save original high-resolution masks and overlay to disk
cv2.imwrite(os.path.join(BASE_DIR, "nrich_mask.png"), nrich_bin * 255)
cv2.imwrite(os.path.join(BASE_DIR, "normal_mask.png"), normal_bin * 255)
cv2.imwrite(os.path.join(BASE_DIR, "overlay_prediction.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

print("💾 Saved high-quality outputs successfully")

# Display ONLY the final overlaid image in original proportions (No titles, no axes)
dpi = 100
fig = plt.figure(figsize=(W / dpi, H / dpi), dpi=dpi)
ax = plt.Axes(fig, [0., 0., 1., 1.])
ax.set_axis_off()
fig.add_axes(ax)
ax.imshow(overlay)
plt.show()